# Lab 1: NumPy for Cat vs Dog Faces

In this notebook, you will treat images as arrays and build a tiny **nearest-centroid baseline** for binary classification.

**Dataset assumption**

Use a prepared subset of the Kaggle **Animal Faces** dataset, keeping only the
**cat** and **dog** face classes, with this structure:

```text
data/cats_dogs_faces_small/
  train/
    cat/
    dog/
  val/
    cat/
    dog/
  test/
    cat/
    dog/
```

The images should already be resized to about **64 x 64** so the lab stays lightweight.

You can build this subset with the helper script in
`scripts/download_animal_faces.py`.

**Questions in this lab**

1. Inspect an image as a NumPy array
2. Normalize pixels and convert RGB to grayscale
3. Crop and flip an image using slicing
4. Extract simple hand-crafted features
5. Build a nearest-centroid classifier
6. Evaluate the baseline and reflect on its limits


In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from lab_utils.visualization import (
    plot_centroid_heatmap,
    plot_feature_vector,
    plot_prediction_gallery,
    show_image_gallery,
)

DATA_ROOT = Path("data/cats_dogs_faces_small")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

LABELS = ("cat", "dog")
SPLITS = ("train", "val", "test")
FEATURE_NAMES = [
    "mean_r",
    "mean_g",
    "mean_b",
    "std_r",
    "std_g",
    "std_b",
    "gray_contrast",
    "center_brightness",
    "hist_0",
    "hist_1",
    "hist_2",
    "hist_3",
    "hist_4",
    "hist_5",
    "hist_6",
    "hist_7",
]

def list_image_paths(split: str) -> list[Path]:
    paths = []
    for label in LABELS:
        label_dir = DATA_ROOT / split / label
        paths.extend(sorted(label_dir.glob("*.jpg")))
        paths.extend(sorted(label_dir.glob("*.png")))
    return sorted(paths)

def load_image(path: Path) -> np.ndarray:
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"))

def label_from_path(path: Path) -> str:
    return path.parent.name

def sample_per_class(paths: list[Path], n_per_class: int) -> list[Path]:
    selected = []
    for label in LABELS:
        label_paths = [path for path in paths if label_from_path(path) == label]
        selected.extend(label_paths[:n_per_class])
    return selected

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Dataset not found. Place the prepared subset at data/cats_dogs_faces_small/."
    )

train_paths = list_image_paths("train")
test_paths = list_image_paths("test")

print(f"Train images: {len(train_paths)}")
print(f"Test images:  {len(test_paths)}")
print(f"Example file: {train_paths[0] if train_paths else 'No files found'}")


### Visual Helper: Preview the Dataset

Before we start writing NumPy logic, use the shared utility below to look at a few training examples from each class.


In [ ]:
preview_paths = sample_per_class(train_paths, n_per_class=3)
preview_images = [load_image(path) for path in preview_paths]
preview_titles = [f"{label_from_path(path)}: {path.name}" for path in preview_paths]
show_image_gallery(
    preview_images,
    titles=preview_titles,
    ncols=3,
    figsize=(9, 6),
    suptitle="Cat and dog face training examples",
)


## Question 1: What does one image look like as an array?

Complete the function below so it returns the key properties of a color image:

- height
- width
- number of channels
- dtype
- minimum pixel value
- maximum pixel value

Then display one sample image.


In [ ]:
sample_path = train_paths[0]
sample_image = load_image(sample_path)

def describe_image(image: np.ndarray) -> tuple[int, int, int, str, int, int]:
    # TODO: replace the placeholder values with real values from the image array
    height = 0
    width = 0
    channels = 0
    dtype_name = ""
    pixel_min = 0
    pixel_max = 0
    return height, width, channels, dtype_name, pixel_min, pixel_max

height, width, channels, dtype_name, pixel_min, pixel_max = describe_image(sample_image)

print("shape:", sample_image.shape)
print("dtype:", dtype_name)
print("range:", pixel_min, "to", pixel_max)

plt.figure(figsize=(3, 3))
plt.imshow(sample_image)
plt.title(sample_path.name)
plt.axis("off")

assert channels == 3, "A color image should have 3 channels."
assert 0 <= pixel_min <= pixel_max <= 255, "Pixel values should stay in the 0-255 range."


## Question 2: Why do we normalize pixels?

Complete these two helper functions:

- `to_float01`: convert an unsigned integer image into floating point values in `[0, 1]`
- `rgb_to_gray`: turn an RGB image into a single grayscale channel using weighted averaging

Use the standard weights `0.299`, `0.587`, and `0.114`.


In [ ]:
def to_float01(image: np.ndarray) -> np.ndarray:
    # TODO: convert to float32 and divide by 255
    raise NotImplementedError("Implement pixel normalization here.")

def rgb_to_gray(image_float: np.ndarray) -> np.ndarray:
    # TODO: compute a weighted grayscale image with shape (H, W)
    raise NotImplementedError("Implement grayscale conversion here.")

sample_float = to_float01(sample_image)
sample_gray = rgb_to_gray(sample_float)

print("float dtype:", sample_float.dtype)
print("float range:", sample_float.min(), "to", sample_float.max())
print("gray shape:", sample_gray.shape)

assert sample_float.dtype == np.float32, "The normalized image should use float32."
assert sample_float.min() >= 0.0 and sample_float.max() <= 1.0, "Normalization should produce values in [0, 1]."
assert sample_gray.shape == sample_image.shape[:2], "A grayscale image should keep only height and width."


## Question 3: Can you crop and flip with slicing?

Complete the functions below:

- `center_crop`: return a centered square crop
- `flip_horizontal`: mirror the image from left to right using slicing only

Try a crop size of `48` on a `64 x 64` image.


In [ ]:
def center_crop(image: np.ndarray, crop_size: int = 48) -> np.ndarray:
    # TODO: compute the crop indices and return the center square
    raise NotImplementedError("Implement a center crop.")

def flip_horizontal(image: np.ndarray) -> np.ndarray:
    # TODO: use NumPy slicing to reverse the width dimension
    raise NotImplementedError("Implement a horizontal flip.")

cropped_image = center_crop(sample_image, crop_size=48)
flipped_image = flip_horizontal(sample_image)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(sample_image)
axes[0].set_title("Original")
axes[1].imshow(cropped_image)
axes[1].set_title("Center Crop")
axes[2].imshow(flipped_image)
axes[2].set_title("Horizontal Flip")

for ax in axes:
    ax.axis("off")

assert cropped_image.shape[0] == 48 and cropped_image.shape[1] == 48, "The crop should be 48 x 48."
assert flipped_image.shape == sample_image.shape, "Flipping should not change the image shape."


## Question 4: How can we turn one image into a feature vector?

Build a small hand-crafted feature vector with:

- mean RGB values
- standard deviation of RGB values
- grayscale contrast (standard deviation)
- mean brightness of the center crop
- an 8-bin grayscale histogram

The final feature vector should have **16 values**.


In [ ]:
def extract_features(image: np.ndarray) -> np.ndarray:
    # Hint: normalize first, then reuse the helper functions from earlier questions.
    image_float = to_float01(image)
    gray = rgb_to_gray(image_float)

    # TODO: compute the remaining features and concatenate them into one 1D array
    raise NotImplementedError("Create the 16-value feature vector here.")

feature_example = extract_features(sample_image)
print("Feature vector length:", feature_example.shape[0])
print(feature_example)

assert feature_example.ndim == 1, "Features should be a 1D vector."
assert feature_example.shape[0] == 16, "Your handcrafted feature vector should contain 16 values."

plot_feature_vector(
    feature_example,
    FEATURE_NAMES,
    title="Hand-crafted feature vector for one image",
)


## Question 5: Train a nearest-centroid baseline

Use the feature vectors to build a simple classifier:

1. Take a small balanced subset of the training images
2. Compute one centroid for cats and one centroid for dogs
3. Predict the label of a test image using the closest centroid

This is not a deep model. It is just a fast NumPy baseline that gives us something to compare against later.


In [ ]:
def build_feature_matrix(paths: list[Path]) -> tuple[np.ndarray, np.ndarray]:
    # TODO: return:
    #   X -> shape (N, 16)
    #   y -> integer labels where cat=0 and dog=1
    raise NotImplementedError("Build the feature matrix from image paths.")

def compute_centroids(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    # TODO: return an array with shape (2, 16)
    raise NotImplementedError("Compute one centroid per class.")

def predict_with_centroids(X: np.ndarray, centroids: np.ndarray) -> np.ndarray:
    # TODO: use Euclidean distance to predict the closest centroid for each sample
    raise NotImplementedError("Predict labels using centroid distances.")

train_subset = sample_per_class(train_paths, n_per_class=80)
test_subset = sample_per_class(test_paths, n_per_class=30)

X_train, y_train = build_feature_matrix(train_subset)
X_test, y_test = build_feature_matrix(test_subset)

centroids = compute_centroids(X_train, y_train)
test_pred = predict_with_centroids(X_test, centroids)

print("Train feature matrix:", X_train.shape)
print("Test feature matrix:", X_test.shape)
print("Centroids:", centroids.shape)

assert X_train.shape[1] == 16, "Each training example should have 16 features."
assert centroids.shape == (2, 16), "There should be one centroid per class."
assert test_pred.shape == y_test.shape, "Predictions should match the number of test labels."

plot_centroid_heatmap(
    centroids,
    FEATURE_NAMES,
    class_names=LABELS,
    title="Nearest-centroid feature profiles",
)


## Question 6: Evaluate and save the baseline predictions

Compute test accuracy, inspect a few mistakes, and save your predictions to:

`artifacts/lab1_numpy_predictions.csv`

Include these columns:

- `filepath`
- `label`
- `pred_numpy`
- `correct_numpy`


In [ ]:
def accuracy_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # TODO: compute the fraction of correct predictions
    raise NotImplementedError("Implement a simple accuracy function.")

test_accuracy = accuracy_score(y_test, test_pred)
print(f"NumPy baseline accuracy: {test_accuracy:.3f}")

label_names = np.array(["cat", "dog"])
mistake_rows = []

for path, true_id, pred_id in zip(test_subset, y_test, test_pred):
    mistake_rows.append(
        {
            "filepath": str(path.relative_to(DATA_ROOT)),
            "label": label_names[true_id],
            "pred_numpy": label_names[pred_id],
            "correct_numpy": bool(true_id == pred_id),
        }
    )

output_path = ARTIFACT_DIR / "lab1_numpy_predictions.csv"
with output_path.open("w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["filepath", "label", "pred_numpy", "correct_numpy"],
    )
    writer.writeheader()
    writer.writerows(mistake_rows)

print(f"Saved predictions to: {output_path}")

plot_prediction_gallery(
    test_subset,
    label_names[y_test],
    label_names[test_pred],
    load_image,
    max_items=8,
    ncols=4,
    figsize=(10, 6),
)

wrong_examples = [row for row in mistake_rows if not row["correct_numpy"]][:5]
wrong_examples


## Reflection

Answer these short questions in your own words:

1. Why do we normalize pixel values before computing features?
2. Why might mean RGB fail on images with cluttered backgrounds?
3. Why does this baseline lose important spatial information?

**Optional extension**

Try one extra feature, such as:

- color channel ratios
- a sharper center crop
- a 16-bin grayscale histogram instead of 8 bins
